# 第7回: CNNRNN - 複数画像と関節角度のマルチモーダル学習

このノートブックでは、**画像（視覚情報）** と **関節角度（固有受容感覚）** を同時に扱う
**CNNRNN（CNN + RNN）** アーキテクチャについて学びます。
Robomimic / LeRobot 形式のデータを用いて、マルチモーダルな時系列予測モデルを構築・学習・評価します。

## 1. CNNRNNの概要

### マルチモーダル学習とは？

ロボットが実世界で動作するとき、複数のセンサ情報を同時に利用します:

- **視覚情報（画像）**: カメラから得られるRGB画像。環境の状態や物体の位置を把握。
- **固有受容感覚（関節角度）**: ロボットの各関節の角度。自身の姿勢を把握。

これら異なるモダリティの情報を統合して学習することを **マルチモーダル学習** と呼びます。

### CNNRNNアーキテクチャ

CNNRNNは以下の構成要素から成り立ちます:

```
画像 xi ──→ [CNN Encoder] ──→ 画像特徴量
                                    ↓
                              [結合 (concat)]  ←── 関節角度 xv
                                    ↓
                                [LSTMCell]  ←── 前時刻の隠れ状態
                                    ↓
                    ┌───────────────┴───────────────┐
                    ↓                               ↓
            [Image Decoder]                 [Joint Decoder]
                    ↓                               ↓
             予測画像 y_image                 予測関節角度 y_joint
```

**ポイント:**
- CNN Encoder で画像を低次元の特徴量に圧縮
- 画像特徴量と関節角度を結合（concatenate）してLSTMに入力
- LSTMの隠れ状態から、画像と関節角度の両方を予測（デコード）
- 複数カメラがある場合は、各カメラの特徴量を結合して利用可能

## 2. セットアップ

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False

# デバイス設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

# 再現性のためシードを固定
torch.manual_seed(42)
np.random.seed(42)

## 3. CNNRNNモデルの定義

CNNRNNモデルは以下の4つのコンポーネントで構成されます:

1. **Image Encoder**: 64x64のRGB画像を低次元特徴量に変換するCNN
2. **LSTMCell**: 画像特徴量と関節角度を結合した入力を処理するリカレントユニット
3. **Joint Decoder**: LSTM隠れ状態から関節角度を予測する全結合層
4. **Image Decoder**: LSTM隠れ状態から画像を再構成する逆畳み込みネットワーク

In [ ]:
class CNNRNN(nn.Module):
    def __init__(self, rec_dim=50, joint_dim=7, feat_dim=10, im_size=64):
        super().__init__()

        # Image Encoder (64x64 -> feat_dim)
        self.encoder_image = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1),   # 64x64 -> 32x32
            nn.Tanh(),
            nn.Conv2d(64, 32, 3, 2, 1),  # 32x32 -> 16x16
            nn.Tanh(),
            nn.Conv2d(32, 16, 3, 2, 1),  # 16x16 -> 8x8
            nn.Tanh(),
            nn.Conv2d(16, 12, 3, 2, 1),  # 8x8 -> 4x4
            nn.Tanh(),
            nn.Conv2d(12, 8, 3, 2, 1),   # 4x4 -> 2x2
            nn.Tanh(),
            nn.Flatten(),
            nn.Linear(8 * 2 * 2, 50),
            nn.Tanh(),
            nn.Linear(50, feat_dim),
            nn.Tanh(),
        )

        # Recurrent (LSTM)
        rec_in = feat_dim + joint_dim
        self.rec = nn.LSTMCell(rec_in, rec_dim)

        # Joint Decoder
        self.decoder_joint = nn.Sequential(
            nn.Linear(rec_dim, joint_dim),
            nn.Tanh()
        )

        # Image Decoder (rec_dim -> 64x64x3)
        self.decoder_image = nn.Sequential(
            nn.Linear(rec_dim, 8 * 2 * 2),
            nn.Tanh(),
            nn.Unflatten(1, (8, 2, 2)),
            nn.ConvTranspose2d(8, 12, 3, 2, 1, 1),   # 2x2 -> 4x4
            nn.Tanh(),
            nn.ConvTranspose2d(12, 16, 3, 2, 1, 1),  # 4x4 -> 8x8
            nn.Tanh(),
            nn.ConvTranspose2d(16, 32, 3, 2, 1, 1),  # 8x8 -> 16x16
            nn.Tanh(),
            nn.ConvTranspose2d(32, 64, 3, 2, 1, 1),  # 16x16 -> 32x32
            nn.Tanh(),
            nn.ConvTranspose2d(64, 3, 3, 2, 1, 1),   # 32x32 -> 64x64
            nn.Tanh(),
        )

    def forward(self, xi, xv, state=None):
        """
        Args:
            xi: 画像入力 (batch, 3, 64, 64)
            xv: 関節角度入力 (batch, joint_dim)
            state: LSTMの隠れ状態 (h, c) のタプル
        Returns:
            y_image: 予測画像 (batch, 3, 64, 64)
            y_joint: 予測関節角度 (batch, joint_dim)
            rnn_hid: 更新されたLSTM隠れ状態
        """
        im_feat = self.encoder_image(xi)
        hid = torch.cat([im_feat, xv], -1)
        rnn_hid = self.rec(hid, state)
        y_joint = self.decoder_joint(rnn_hid[0])
        y_image = self.decoder_image(rnn_hid[0])
        return y_image, y_joint, rnn_hid

### モデルの動作確認

ダミーデータを使ってモデルのフォワードパスを確認します。

In [ ]:
# モデルの初期化
model = CNNRNN(rec_dim=50, joint_dim=7, feat_dim=10, im_size=64)

# ダミー入力
batch_size = 4
dummy_img = torch.randn(batch_size, 3, 64, 64)
dummy_joint = torch.randn(batch_size, 7)

# フォワードパス
y_img, y_joint, state = model(dummy_img, dummy_joint)

print(f"入力画像:     {dummy_img.shape}")
print(f"入力関節角度:  {dummy_joint.shape}")
print(f"予測画像:     {y_img.shape}")
print(f"予測関節角度:  {y_joint.shape}")
print(f"LSTM h状態:   {state[0].shape}")
print(f"LSTM c状態:   {state[1].shape}")
print(f"\nパラメータ数: {sum(p.numel() for p in model.parameters()):,}")

## 4. BPTT（Backpropagation Through Time）トレーナー

マルチモーダル学習では、**画像再構成損失** と **関節角度予測損失** の2つの損失関数を組み合わせます。

`loss_weights` パラメータにより、各モダリティの損失の重みを調整できます:
- `loss_weights[0]`: 画像損失の重み
- `loss_weights[1]`: 関節角度損失の重み

時系列データに対してBPTT（時間方向の逆伝播）を行い、LSTMの隠れ状態を時間ステップ間で引き継ぎます。

In [ ]:
class fullBPTTtrainer:
    def __init__(self, model, optimizer, loss_weights=[1.0, 1.0], device="cpu"):
        self.device = device
        self.optimizer = optimizer
        self.loss_weights = loss_weights
        self.model = model.to(device)

    def process_epoch(self, data, training=True):
        if training:
            self.model.train()
        else:
            self.model.eval()

        total_loss = 0.0
        for n_batch, ((x_img, x_joint), (y_img, y_joint)) in enumerate(data):
            x_img = x_img.to(self.device)
            y_img = y_img.to(self.device)
            x_joint = x_joint.to(self.device)
            y_joint = y_joint.to(self.device)

            state = None
            yi_list, yv_list = [], []
            self.optimizer.zero_grad()

            # 時間ステップごとにフォワードパスを実行
            for t in range(x_img.shape[1] - 1):
                _yi, _yv, state = self.model(x_img[:, t], x_joint[:, t], state)
                yi_list.append(_yi)
                yv_list.append(_yv)

            # 予測結果をスタック
            yi_hat = torch.stack(yi_list, dim=1)
            yv_hat = torch.stack(yv_list, dim=1)

            # マルチモーダル損失の計算
            img_loss = nn.MSELoss()(yi_hat, y_img[:, 1:]) * self.loss_weights[0]
            joint_loss = nn.MSELoss()(yv_hat, y_joint[:, 1:]) * self.loss_weights[1]
            loss = img_loss + joint_loss
            total_loss += loss.item()

            if training:
                loss.backward()
                self.optimizer.step()

        return total_loss / (n_batch + 1)

## 5. データセット（LeRobot形式）

LeRobot / Robomimic 形式のデータを読み込むためのデータセットクラスを定義します。

データ構造:
```
data_dir/
├── observation.state.npy          # 関節角度 (num_episodes, seq_len, joint_dim)
└── observation.images.agentview/
    ├── episode_000000.npy         # 画像 (seq_len, H, W, 3)
    ├── episode_000001.npy
    └── ...
```

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.states = np.load(os.path.join(data_dir, "observation.state.npy"))
        self.num_episodes = len(self.states)
        self.data_dir = data_dir

    def __len__(self):
        return self.num_episodes

    def __getitem__(self, idx):
        # 画像の読み込みと前処理
        images = np.load(os.path.join(
            self.data_dir, "observation.images.agentview",
            f"episode_{idx:06d}.npy")).astype(np.float32) / 255.0
        # (seq_len, H, W, 3) -> (seq_len, 3, H, W)
        images = torch.from_numpy(images.transpose(0, 3, 1, 2))
        states = torch.from_numpy(self.states[idx].astype(np.float32))

        # 入力と教師信号は同一（trainer側でシフトを処理）
        return (images, states), (images, states)

## 6. ダミーデータの生成と学習

実際のRobomimicデータがない場合でも動作確認ができるよう、ダミーデータを生成します。
ここでは、正弦波ベースの関節角度とランダムなグラデーション画像を生成します。

In [ ]:
def create_dummy_data(data_dir, num_episodes=8, seq_len=20, joint_dim=7, im_size=64):
    """ダミーのマルチモーダルデータを生成する"""
    os.makedirs(os.path.join(data_dir, "observation.images.agentview"), exist_ok=True)

    # 関節角度: 正弦波ベースの軌跡
    all_states = []
    for i in range(num_episodes):
        t = np.linspace(0, 2 * np.pi, seq_len)
        states = np.stack([
            np.sin(t + j * 0.5) * 0.5 for j in range(joint_dim)
        ], axis=-1)
        # エピソードごとに少しランダム性を加える
        states += np.random.randn(*states.shape) * 0.05
        all_states.append(states)

    all_states = np.array(all_states, dtype=np.float32)
    np.save(os.path.join(data_dir, "observation.state.npy"), all_states)

    # 画像: 関節角度に基づくグラデーション画像
    for i in range(num_episodes):
        images = []
        for t_step in range(seq_len):
            img = np.zeros((im_size, im_size, 3), dtype=np.uint8)
            for c in range(3):
                val = int((all_states[i, t_step, c] + 1.0) / 2.0 * 255)
                val = np.clip(val, 0, 255)
                # グラデーションパターンを生成
                grad = np.linspace(0, val, im_size).reshape(-1, 1)
                img[:, :, c] = np.tile(grad, (1, im_size)).astype(np.uint8)
            images.append(img)
        images = np.array(images, dtype=np.uint8)
        np.save(os.path.join(
            data_dir, "observation.images.agentview",
            f"episode_{i:06d}.npy"), images)

    print(f"ダミーデータ生成完了:")
    print(f"  エピソード数: {num_episodes}")
    print(f"  シーケンス長: {seq_len}")
    print(f"  関節角度次元: {joint_dim}")
    print(f"  画像サイズ:   {im_size}x{im_size}")
    return data_dir

# ダミーデータの生成
data_dir = "/tmp/cnnrnn_dummy_data"
create_dummy_data(data_dir, num_episodes=8, seq_len=20, joint_dim=7)

## 7. モデルの学習

ダミーデータを使ってCNNRNNモデルを学習します。

In [ ]:
# データセットとデータローダーの作成
dataset = MultimodalDataset(data_dir)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# モデル・最適化器・トレーナーの初期化
model = CNNRNN(rec_dim=50, joint_dim=7, feat_dim=10)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trainer = fullBPTTtrainer(model, optimizer, loss_weights=[1.0, 1.0], device=device)

# 学習ループ
num_epochs = 50
loss_history = []

print("学習開始...")
for epoch in range(num_epochs):
    loss = trainer.process_epoch(dataloader, training=True)
    loss_history.append(loss)
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/{num_epochs} | Loss: {loss:.6f}")

print("学習完了!")

## 8. 結果の可視化

### 8.1 損失曲線

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history, linewidth=2)
plt.xlabel("エポック", fontsize=12)
plt.ylabel("損失 (MSE)", fontsize=12)
plt.title("学習損失の推移", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 8.2 入力画像 vs 予測画像の比較

In [ ]:
# テストデータで予測
model.eval()
(test_imgs, test_joints), _ = dataset[0]
test_imgs = test_imgs.unsqueeze(0).to(device)
test_joints = test_joints.unsqueeze(0).to(device)

pred_imgs, pred_joints = [], []
state = None
with torch.no_grad():
    for t in range(test_imgs.shape[1] - 1):
        yi, yv, state = model(test_imgs[:, t], test_joints[:, t], state)
        pred_imgs.append(yi.cpu())
        pred_joints.append(yv.cpu())

pred_imgs = torch.cat(pred_imgs, dim=0)   # (T-1, 3, 64, 64)
pred_joints = torch.cat(pred_joints, dim=0)  # (T-1, joint_dim)

# 画像比較（最初の8タイムステップ）
n_show = min(8, pred_imgs.shape[0])
fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 5))

for t in range(n_show):
    # 入力画像（次ステップ = 教師信号）
    gt = test_imgs[0, t + 1].cpu().numpy().transpose(1, 2, 0)
    gt = np.clip((gt + 1) / 2, 0, 1)  # [-1,1] -> [0,1]
    axes[0, t].imshow(gt)
    axes[0, t].set_title(f"t={t+1}", fontsize=9)
    axes[0, t].axis("off")

    # 予測画像
    pr = pred_imgs[t].numpy().transpose(1, 2, 0)
    pr = np.clip((pr + 1) / 2, 0, 1)
    axes[1, t].imshow(pr)
    axes[1, t].axis("off")

axes[0, 0].set_ylabel("正解", fontsize=12)
axes[1, 0].set_ylabel("予測", fontsize=12)
fig.suptitle("入力画像 vs 予測画像", fontsize=14)
plt.tight_layout()
plt.show()

### 8.3 関節角度の予測結果

In [ ]:
gt_joints = test_joints[0, 1:].cpu().numpy()  # (T-1, joint_dim)
pr_joints = pred_joints.numpy()

joint_dim = gt_joints.shape[1]
fig, axes = plt.subplots(joint_dim, 1, figsize=(10, 2 * joint_dim), sharex=True)

for j in range(joint_dim):
    axes[j].plot(gt_joints[:, j], "b-", label="正解", linewidth=2)
    axes[j].plot(pr_joints[:, j], "r--", label="予測", linewidth=2)
    axes[j].set_ylabel(f"関節 {j}", fontsize=10)
    axes[j].grid(True, alpha=0.3)
    if j == 0:
        axes[j].legend(fontsize=9)

axes[-1].set_xlabel("タイムステップ", fontsize=12)
fig.suptitle("関節角度: 正解 vs 予測", fontsize=14)
plt.tight_layout()
plt.show()

## 9. loss_weightsの影響

画像損失と関節角度損失の重みバランスがモデルの学習にどう影響するかを確認します。

In [ ]:
# 異なるloss_weightsで学習を比較
weight_configs = {
    "画像重視 [10.0, 1.0]": [10.0, 1.0],
    "均等 [1.0, 1.0]": [1.0, 1.0],
    "関節重視 [1.0, 10.0]": [1.0, 10.0],
}

results = {}
num_epochs_cmp = 30

for name, weights in weight_configs.items():
    print(f"\n--- {name} ---")
    m = CNNRNN(rec_dim=50, joint_dim=7, feat_dim=10)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    tr = fullBPTTtrainer(m, opt, loss_weights=weights, device=device)

    losses = []
    for epoch in range(num_epochs_cmp):
        loss = tr.process_epoch(dataloader, training=True)
        losses.append(loss)

    results[name] = losses
    print(f"  最終損失: {losses[-1]:.6f}")

# 比較プロット
plt.figure(figsize=(8, 4))
for name, losses in results.items():
    plt.plot(losses, label=name, linewidth=2)
plt.xlabel("エポック", fontsize=12)
plt.ylabel("損失", fontsize=12)
plt.title("loss_weightsによる学習曲線の比較", fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 演習問題

### 演習1: CNNRNNのエンコーダ・デコーダ部分を実装せよ

以下のテンプレートに従って、`MyCNNRNN` クラスの `encoder_image` と `decoder_image` を実装してください。

**要件:**
- エンコーダ: 64x64のRGB画像を `feat_dim` 次元の特徴量に変換
- デコーダ: `rec_dim` 次元のLSTM隠れ状態から64x64のRGB画像を再構成
- 活性化関数は `Tanh` を使用

In [ ]:
class MyCNNRNN(nn.Module):
    def __init__(self, rec_dim=50, joint_dim=7, feat_dim=10, im_size=64):
        super().__init__()

        # TODO: Image Encoder を実装してください
        self.encoder_image = nn.Sequential(
            # ここにコードを書いてください
        )

        # Recurrent (LSTM)
        rec_in = feat_dim + joint_dim
        self.rec = nn.LSTMCell(rec_in, rec_dim)

        # Joint Decoder
        self.decoder_joint = nn.Sequential(
            nn.Linear(rec_dim, joint_dim),
            nn.Tanh()
        )

        # TODO: Image Decoder を実装してください
        self.decoder_image = nn.Sequential(
            # ここにコードを書いてください
        )

    def forward(self, xi, xv, state=None):
        im_feat = self.encoder_image(xi)
        hid = torch.cat([im_feat, xv], -1)
        rnn_hid = self.rec(hid, state)
        y_joint = self.decoder_joint(rnn_hid[0])
        y_image = self.decoder_image(rnn_hid[0])
        return y_image, y_joint, rnn_hid

# テスト
my_model = MyCNNRNN()
test_x_img = torch.randn(2, 3, 64, 64)
test_x_jnt = torch.randn(2, 7)
yi, yv, s = my_model(test_x_img, test_x_jnt)
assert yi.shape == (2, 3, 64, 64), f"画像出力のshapeが不正: {yi.shape}"
assert yv.shape == (2, 7), f"関節角度出力のshapeが不正: {yv.shape}"
print("テスト合格!")

<details>
<summary><b>回答例を表示</b></summary>

```python
class MyCNNRNN(nn.Module):
    def __init__(self, rec_dim=50, joint_dim=7, feat_dim=10, im_size=64):
        super().__init__()

        # Image Encoder (64x64 -> feat_dim)
        self.encoder_image = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1),   # 64x64 -> 32x32
            nn.Tanh(),
            nn.Conv2d(64, 32, 3, 2, 1),  # 32x32 -> 16x16
            nn.Tanh(),
            nn.Conv2d(32, 16, 3, 2, 1),  # 16x16 -> 8x8
            nn.Tanh(),
            nn.Conv2d(16, 12, 3, 2, 1),  # 8x8 -> 4x4
            nn.Tanh(),
            nn.Conv2d(12, 8, 3, 2, 1),   # 4x4 -> 2x2
            nn.Tanh(),
            nn.Flatten(),
            nn.Linear(8 * 2 * 2, 50),
            nn.Tanh(),
            nn.Linear(50, feat_dim),
            nn.Tanh(),
        )

        rec_in = feat_dim + joint_dim
        self.rec = nn.LSTMCell(rec_in, rec_dim)

        self.decoder_joint = nn.Sequential(
            nn.Linear(rec_dim, joint_dim),
            nn.Tanh()
        )

        # Image Decoder (rec_dim -> 64x64x3)
        self.decoder_image = nn.Sequential(
            nn.Linear(rec_dim, 8 * 2 * 2),
            nn.Tanh(),
            nn.Unflatten(1, (8, 2, 2)),
            nn.ConvTranspose2d(8, 12, 3, 2, 1, 1),   # 2x2 -> 4x4
            nn.Tanh(),
            nn.ConvTranspose2d(12, 16, 3, 2, 1, 1),  # 4x4 -> 8x8
            nn.Tanh(),
            nn.ConvTranspose2d(16, 32, 3, 2, 1, 1),  # 8x8 -> 16x16
            nn.Tanh(),
            nn.ConvTranspose2d(32, 64, 3, 2, 1, 1),  # 16x16 -> 32x32
            nn.Tanh(),
            nn.ConvTranspose2d(64, 3, 3, 2, 1, 1),   # 32x32 -> 64x64
            nn.Tanh(),
        )

    def forward(self, xi, xv, state=None):
        im_feat = self.encoder_image(xi)
        hid = torch.cat([im_feat, xv], -1)
        rnn_hid = self.rec(hid, state)
        y_joint = self.decoder_joint(rnn_hid[0])
        y_image = self.decoder_image(rnn_hid[0])
        return y_image, y_joint, rnn_hid

```

</details>

### 演習2: fullBPTTtrainerのprocess_epochを実装し、マルチモーダル学習を実行せよ

以下のテンプレートに従って、`MyTrainer` クラスの `process_epoch` メソッドを実装してください。

**要件:**
- 各バッチについて、時間ステップごとにフォワードパスを実行
- 画像損失と関節角度損失を `loss_weights` で重み付けして合算
- `training=True` の場合のみ逆伝播と重み更新を行う
- エポック全体の平均損失を返す

In [ ]:
class MyTrainer:
    def __init__(self, model, optimizer, loss_weights=[1.0, 1.0], device="cpu"):
        self.device = device
        self.optimizer = optimizer
        self.loss_weights = loss_weights
        self.model = model.to(device)

    def process_epoch(self, data, training=True):
        # TODO: ここにコードを書いてください
        # ヒント:
        # 1. model.train() / model.eval() の切り替え
        # 2. バッチループ
        # 3. 時間ステップループ (x_img.shape[1] - 1 回)
        # 4. 予測結果のスタック
        # 5. 画像損失 + 関節角度損失の計算
        # 6. training時のみ逆伝播
        pass

# テスト
test_model = CNNRNN(rec_dim=50, joint_dim=7, feat_dim=10)
test_opt = torch.optim.Adam(test_model.parameters(), lr=1e-3)
my_trainer = MyTrainer(test_model, test_opt, loss_weights=[1.0, 1.0], device=device)

loss = my_trainer.process_epoch(dataloader, training=True)
print(f"1エポックの損失: {loss:.6f}")
print("テスト合格!")

<details>
<summary><b>回答例を表示</b></summary>

```python
class MyTrainer:
    def __init__(self, model, optimizer, loss_weights=[1.0, 1.0], device="cpu"):
        self.device = device
        self.optimizer = optimizer
        self.loss_weights = loss_weights
        self.model = model.to(device)

    def process_epoch(self, data, training=True):
        if training:
            self.model.train()
        else:
            self.model.eval()

        total_loss = 0.0
        for n_batch, ((x_img, x_joint), (y_img, y_joint)) in enumerate(data):
            x_img = x_img.to(self.device)
            y_img = y_img.to(self.device)
            x_joint = x_joint.to(self.device)
            y_joint = y_joint.to(self.device)

            state = None
            yi_list, yv_list = [], []
            self.optimizer.zero_grad()

            for t in range(x_img.shape[1] - 1):
                _yi, _yv, state = self.model(x_img[:, t], x_joint[:, t], state)
                yi_list.append(_yi)
                yv_list.append(_yv)

            yi_hat = torch.stack(yi_list, dim=1)
            yv_hat = torch.stack(yv_list, dim=1)

            img_loss = nn.MSELoss()(yi_hat, y_img[:, 1:]) * self.loss_weights[0]
            joint_loss = nn.MSELoss()(yv_hat, y_joint[:, 1:]) * self.loss_weights[1]
            loss = img_loss + joint_loss
            total_loss += loss.item()

            if training:
                loss.backward()
                self.optimizer.step()

        return total_loss / (n_batch + 1)

```

</details>

### 演習3: loss_weightsを変えて画像再構成と関節角度予測のトレードオフを調べよ

以下の5つの `loss_weights` 設定でそれぞれ学習を行い、各設定での:
1. 画像再構成の品質（画像MSE）
2. 関節角度予測の精度（関節角度MSE）

を別々に計測して比較してください。

```python
configs = [
    [100.0, 1.0],  # 画像を強く重視
    [10.0, 1.0],   # 画像をやや重視
    [1.0, 1.0],    # 均等
    [1.0, 10.0],   # 関節角度をやや重視
    [1.0, 100.0],  # 関節角度を強く重視
]
```

結果をグラフ（散布図など）で可視化し、トレードオフの関係を考察してください。

In [ ]:
configs = [
    [100.0, 1.0],
    [10.0, 1.0],
    [1.0, 1.0],
    [1.0, 10.0],
    [1.0, 100.0],
]

# ここにコードを書いてください
# ヒント:
# 1. 各configで新しいモデルとトレーナーを作成
# 2. 学習後に画像損失と関節角度損失を個別に計算
# 3. 結果を散布図で可視化

<details>
<summary><b>回答例を表示</b></summary>

```python
configs = [
    [100.0, 1.0],
    [10.0, 1.0],
    [1.0, 1.0],
    [1.0, 10.0],
    [1.0, 100.0],
]

img_losses = []
jnt_losses = []
labels = []
num_ep = 30

for weights in configs:
    label = f"[{weights[0]}, {weights[1]}]"
    print(f"学習中: loss_weights = {label}")

    m = CNNRNN(rec_dim=50, joint_dim=7, feat_dim=10)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    tr = fullBPTTtrainer(m, opt, loss_weights=weights, device=device)

    for ep in range(num_ep):
        tr.process_epoch(dataloader, training=True)

    # 個別の損失を計測
    m.eval()
    total_img_loss = 0.0
    total_jnt_loss = 0.0
    count = 0
    with torch.no_grad():
        for (x_img, x_jnt), (y_img, y_jnt) in dataloader:
            x_img, x_jnt = x_img.to(device), x_jnt.to(device)
            y_img, y_jnt = y_img.to(device), y_jnt.to(device)
            state = None
            yi_list, yv_list = [], []
            for t in range(x_img.shape[1] - 1):
                _yi, _yv, state = m(x_img[:, t], x_jnt[:, t], state)
                yi_list.append(_yi)
                yv_list.append(_yv)
            yi_hat = torch.stack(yi_list, dim=1)
            yv_hat = torch.stack(yv_list, dim=1)
            total_img_loss += nn.MSELoss()(yi_hat, y_img[:, 1:]).item()
            total_jnt_loss += nn.MSELoss()(yv_hat, y_jnt[:, 1:]).item()
            count += 1

    img_losses.append(total_img_loss / count)
    jnt_losses.append(total_jnt_loss / count)
    labels.append(label)

# 散布図で可視化
plt.figure(figsize=(8, 6))
for i, label in enumerate(labels):
    plt.scatter(img_losses[i], jnt_losses[i], s=100, zorder=5)
    plt.annotate(label, (img_losses[i], jnt_losses[i]),
                textcoords="offset points", xytext=(10, 5), fontsize=9)

plt.xlabel("画像再構成MSE", fontsize=12)
plt.ylabel("関節角度予測MSE", fontsize=12)
plt.title("loss_weightsによるトレードオフ", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n考察: 画像の重みを大きくすると画像再構成の品質は向上するが、")
print("関節角度の予測精度は低下する（トレードオフの関係）。")

```

</details>

---
## まとめ

このノートブックでは以下の内容を学びました:

1. **CNNRNNアーキテクチャ**: CNN（画像エンコーダ/デコーダ）とRNN（LSTM）を組み合わせたマルチモーダルモデル
2. **マルチモーダル学習**: 画像と関節角度の2つのモダリティを同時に扱う学習手法
3. **BPTT（時間方向の逆伝播）**: 時系列データに対するLSTMの学習手法
4. **損失関数の重み付け**: `loss_weights` による画像再構成と関節角度予測のバランス調整
5. **LeRobot形式のデータ**: Robomimicデータセットの読み込みと前処理

**次のステップ:**
- 実際のRobomimicデータを使った学習
- 複数カメラ画像の統合（複数の画像エンコーダの出力を結合）
- Attention機構の導入による性能向上